# Thesis Figure Regeneration (Vector)

**Source script:** `scripts/regenerate_thesis_figures_vector.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

ROOT = Path('/Users/omaralneser/Documents/TYP')
DATASET = ROOT / 'outputs/eda_v3_d7_compare/datasets/d7_second_only.csv'
EDA = ROOT / 'outputs/eda_v3_d7_compare/d7_second_only/eda_v3'
TABLES = EDA / 'tables'
UNSUP_TABLES = EDA / 'unsupervised/tables'
FIG_OUT = ROOT / 'thesis-paper/TYP_template/figures'

plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})

DISEASE_ORDER = [
    'acute pancreatitis',
    'acute flare-up of triaditis/chronic enteropathy',
    'acute gastroenteritis',
    'parvovirus enteritis',
]
DISEASE_TOKEN = {
    'acute pancreatitis': 'ap',
    'acute flare-up of triaditis/chronic enteropathy': 'tri',
    'acute gastroenteritis': 'ge',
    'parvovirus enteritis': 'parvo',
}


def save(fig: plt.Figure, name: str) -> None:
    FIG_OUT.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_OUT / name, format='pdf', bbox_inches='tight')
    plt.close(fig)


def pretty_feature_label(name: str) -> str:
    parts = name.split('__')
    base = parts[0].replace('_', ' ')
    suffix = parts[1] if len(parts) > 1 else ''
    suffix = suffix.replace('_', ' ')
    if suffix:
        return f'{base}\\n{suffix}'
    return base


def load_env_frame() -> tuple[pd.DataFrame, np.ndarray, np.ndarray, list[str], PCA]:
    df = pd.read_csv(DATASET)
    if 'moon_phase_sin' not in df.columns or 'moon_phase_cos' not in df.columns:
        if 'moon_phase_angle_deg' in df.columns:
            ang = pd.to_numeric(df['moon_phase_angle_deg'], errors='coerce')
            rad = np.deg2rad(ang)
            df['moon_phase_sin'] = np.sin(rad)
            df['moon_phase_cos'] = np.cos(rad)
    kept = pd.read_csv(TABLES / 'weather_features_kept_after_correlation.csv')['kept_feature'].tolist()
    X = df[kept].apply(pd.to_numeric, errors='coerce').fillna(df[kept].median(numeric_only=True))
    y = df['group'].astype(str).values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    pca = PCA(n_components=6)
    X_pca = pca.fit_transform(X_scaled)
    return df, X_scaled, X_pca, kept, pca


def plot_corr_heatmap(df: pd.DataFrame) -> None:
    corr = pd.read_csv(TABLES / 'weather_correlation_matrix_abs.csv', index_col=0)
    cols_in = [c for c in corr.columns if c in df.columns]
    variances = df[cols_in].apply(pd.to_numeric, errors='coerce').var().sort_values(ascending=False)
    top_cols = variances.head(40).index.tolist()
    sub = corr.loc[top_cols, top_cols]

    fig, ax = plt.subplots(figsize=(9.0, 7.8))
    y = np.arange(sub.shape[0] + 1)
    x = np.arange(sub.shape[1] + 1)
    im = ax.pcolormesh(x, y, sub.values, cmap='Greys', vmin=0.0, vmax=1.0, shading='flat')
    ax.set_xlim(0, sub.shape[1])
    ax.set_ylim(0, sub.shape[0])
    ax.invert_yaxis()
    tick_idx = np.arange(0, len(top_cols), 5)
    ax.set_xticks(tick_idx + 0.5)
    ax.set_yticks(tick_idx + 0.5)
    ax.set_xticklabels([pretty_feature_label(top_cols[i]) for i in tick_idx], rotation=90, va='top')
    ax.set_yticklabels([pretty_feature_label(top_cols[i]) for i in tick_idx])
    ax.set_title('Absolute correlation heatmap (top-variance weather features)')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('|r|')
    save(fig, 'corr_heatmap_weather_topvariance.pdf')


def plot_corr_filter_summary() -> None:
    summary = pd.read_csv(TABLES / 'weather_correlation_filter_summary.csv')
    values = dict(zip(summary['metric'], summary['value']))
    kept = int(values.get('n_weather_numeric_after', 0))
    dropped = int(values.get('n_dropped_high_corr', 0))

    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    bars = ax.bar(['Kept', 'Dropped'], [kept, dropped], color=['#4d4d4d', '#bdbdbd'], edgecolor='black')
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1, f'{int(b.get_height())}',
                ha='center', va='bottom')
    ax.set_ylabel('Feature count')
    ax.set_title('Weather numeric features after correlation filtering')
    ax.set_ylim(0, max(kept, dropped) * 1.15)
    save(fig, 'corr_filter_kept_dropped_summary.pdf')


def plot_pca_scree() -> None:
    pca_var = pd.read_csv(UNSUP_TABLES / 'pca_variance_environmental_only.csv')
    pca_var = pca_var.sort_values('component', key=lambda s: s.str.extract(r'(\d+)').astype(int)[0])
    comp_idx = np.arange(1, len(pca_var) + 1)

    fig, ax1 = plt.subplots(figsize=(7.0, 4.6))
    ax1.bar(comp_idx, pca_var['explained_variance_ratio'], color='#9e9e9e', edgecolor='black')
    ax1.set_xlabel('Principal component')
    ax1.set_ylabel('Explained variance ratio')

    ax2 = ax1.twinx()
    ax2.plot(comp_idx, pca_var['cumulative_variance_ratio'], color='black', marker='o', linewidth=1.5)
    ax2.set_ylabel('Cumulative variance ratio')
    ax2.set_ylim(0, 1.0)

    ax1.set_title('Environmental-only PCA scree and cumulative variance')
    save(fig, 'pca_scree_env.pdf')


def plot_pca_scatter(X_pca: np.ndarray, y: np.ndarray) -> None:
    marker_map = {
        'acute pancreatitis': 'o',
        'acute flare-up of triaditis/chronic enteropathy': 's',
        'acute gastroenteritis': '^',
        'parvovirus enteritis': 'D',
    }
    label_map = {
        'acute pancreatitis': 'acute pancreatitis',
        'acute flare-up of triaditis/chronic enteropathy': 'triaditis/CE flare-up',
        'acute gastroenteritis': 'acute gastroenteritis',
        'parvovirus enteritis': 'parvovirus enteritis',
    }

    fig, ax = plt.subplots(figsize=(8.0, 5.6))
    for d in DISEASE_ORDER:
        mask = y == d
        ax.scatter(
            X_pca[mask, 0], X_pca[mask, 1],
            marker=marker_map[d],
            facecolors='none',
            edgecolors='black',
            s=28,
            linewidths=0.8,
            alpha=0.9,
            label=label_map[d],
        )
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title('Environmental-only PCA (PC1 vs PC2) by diagnosis')
    ax.legend(frameon=True, facecolor='white', edgecolor='black', loc='upper right')
    ax.grid(True, color='#d9d9d9', linestyle='--', linewidth=0.6)
    save(fig, 'pca_pc1_pc2_env.pdf')


def compute_kbest() -> int:
    sil = pd.read_csv(UNSUP_TABLES / 'kmeans_silhouette_environmental_only_Dfixed6.csv')
    row = sil.loc[sil['silhouette_score'].idxmax()]
    return int(row['k'])


def plot_kmeans(X_pca: np.ndarray, n_clusters: int, outname: str, title: str) -> None:
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=50)
    labels = km.fit_predict(X_pca[:, :6])

    markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '<', '>', '*']
    fig, ax = plt.subplots(figsize=(8.0, 5.6))
    for c in sorted(np.unique(labels)):
        m = labels == c
        mk = markers[c % len(markers)]
        ax.scatter(
            X_pca[m, 0], X_pca[m, 1],
            marker=mk,
            facecolors='none',
            edgecolors='black',
            s=26,
            linewidths=0.8,
            label=f'C{c}'
        )
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(title)
    ax.legend(frameon=True, facecolor='white', edgecolor='black', loc='upper right', ncol=2)
    ax.grid(True, color='#d9d9d9', linestyle='--', linewidth=0.6)
    save(fig, outname)


def plot_hclust(X_pca: np.ndarray) -> None:
    Z = linkage(X_pca[:, :6], method='ward')
    fig, ax = plt.subplots(figsize=(12.0, 4.8))
    dendrogram(
        Z,
        no_labels=True,
        color_threshold=0,
        above_threshold_color='black',
        count_sort='ascending',
        ax=ax,
    )
    for coll in ax.collections:
        coll.set_linewidth(0.7)
    ax.set_xlabel('Samples (ordered leaves)')
    ax.set_ylabel('Ward linkage distance')
    ax.set_title('Hierarchical clustering dendrogram (environmental-only, Dfixed6)')
    ax.grid(True, axis='y', color='#d9d9d9', linestyle='--', linewidth=0.6)
    ax.margins(x=0.0)
    save(fig, 'hclust_dendrogram_env_dfixed6.pdf')


def plot_shap_env() -> None:
    shap = pd.read_csv(TABLES / 'shap_top20_binary.csv')
    env = shap[shap['feature_block'] == 'environmental_only'].copy()

    for disease in DISEASE_ORDER:
        ddf = env[env['disease'] == disease].sort_values('rank_within_group').head(15)
        ddf = ddf.iloc[::-1]
        fig, ax = plt.subplots(figsize=(7.8, 5.6))
        ax.barh(ddf['feature'].str.replace('_', ' '), ddf['mean_abs_shap'], color='#8c8c8c', edgecolor='black')
        ax.set_xlabel('Mean |SHAP|')
        ax.set_ylabel('Feature')
        ax.set_title(f"SHAP feature ranking ({disease})")
        ax.tick_params(axis='y', labelsize=8)
        save(fig, f"shap_{DISEASE_TOKEN[disease]}_env.pdf")


def plot_perm_env() -> None:
    dist = pd.read_csv(TABLES / 'permtest_envonly_auc_distribution.csv')
    summary = pd.read_csv(TABLES / 'permtest_envonly_auc_summary.csv').set_index('disease')

    for disease in DISEASE_ORDER:
        d = dist[dist['disease'] == disease]['perm_mean_auc']
        obs = float(summary.loc[disease, 'observed_auc'])
        pval = float(summary.loc[disease, 'p_value'])

        fig, ax = plt.subplots(figsize=(7.2, 4.8))
        ax.hist(d, bins=24, color='#bdbdbd', edgecolor='black')
        ax.axvline(obs, color='black', linestyle='--', linewidth=2)
        ax.set_xlabel('Permutation mean ROC-AUC')
        ax.set_ylabel('Count')
        ax.set_title(f"Permutation test ({disease}; p={pval:.4f})")
        save(fig, f"permtest_{DISEASE_TOKEN[disease]}_env.pdf")


def plot_monthly(df: pd.DataFrame) -> None:
    date_col = 'presentation_date' if 'presentation_date' in df.columns else 'date of presentation'
    dt = pd.to_datetime(df[date_col], errors='coerce')
    month_counts = dt.dt.month.value_counts().reindex(range(1, 13), fill_value=0)
    labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    ax.bar(labels, month_counts.values, color='#9e9e9e', edgecolor='black')
    ax.set_xlabel('Month')
    ax.set_ylabel('Case count')
    ax.set_title('Monthly distribution of presentations')
    save(fig, 'monthly_case_distribution.pdf')


def main() -> None:
    df, _, X_pca, _, _ = load_env_frame()
    y = df['group'].astype(str).values

    plot_corr_heatmap(df)
    plot_corr_filter_summary()
    plot_pca_scree()
    plot_pca_scatter(X_pca, y)

    kbest = compute_kbest()
    plot_kmeans(X_pca, kbest, 'kmeans_env_kbest_dfixed6.pdf', f'Environmental-only KMeans (Dfixed6, k_best={kbest})')
    plot_kmeans(X_pca, 6, 'kmeans_env_k6_dfixed6.pdf', 'Environmental-only KMeans (Dfixed6, k=6)')
    plot_hclust(X_pca)

    plot_shap_env()
    plot_perm_env()
    plot_monthly(df)


if __name__ == '__main__':
    main()
